# Package

In [19]:
# ----------------------------
# Core
# ----------------------------
from pathlib import Path
import numpy as np
import pandas as pd
from dateutil.relativedelta import relativedelta
import pickle

# ----------------------------
# Feature Store
# ----------------------------
from feast import FeatureStore

# Model
from statsmodels.tsa.ar_model import AutoReg
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Importation des données

In [20]:
# ----------------------------
# Locate Feast repo (notebook-safe)
# ----------------------------
def find_project_root(start: Path, marker: str = "2_data_processing") -> Path:
    p = start.resolve()
    for parent in [p] + list(p.parents):
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(
        f"Impossible de trouver la racine projet (marker '{marker}') depuis {start}"
    )

PROJECT_ROOT = find_project_root(Path.cwd(), marker="2_data_processing")

FEAST_REPO_PATH = (
    PROJECT_ROOT
    / "2_data_processing"
    / "feature_store"
    / "feast_repo"
    / "feature_repo"
)

print("FEAST_REPO_PATH:", FEAST_REPO_PATH)
print("feature_store.yaml exists:", (FEAST_REPO_PATH / "feature_store.yaml").exists())


# ----------------------------
# Load features from Feast
# ----------------------------
def load_features_from_feast(entity_df: pd.DataFrame, feature_refs: list[str]) -> pd.DataFrame:
    fs = FeatureStore(repo_path=str(FEAST_REPO_PATH))
    return fs.get_historical_features(entity_df=entity_df, features=feature_refs).to_df()


# ----------------------------
# Config data
# ----------------------------
START = "1959-01-01"
END   = "2025-09-01"
FREQ  = "MS"
SERIES_ID = "UNRATE"
FEATURE_REFS = ["stationary_value:value"]  # UNRATE déjà stationnaire

dates = pd.date_range(start=START, end=END, freq=FREQ)
entity_df = pd.DataFrame({"series_id": [SERIES_ID] * len(dates), "date": dates})

ts_raw = load_features_from_feast(entity_df, FEATURE_REFS)

ts = (
    ts_raw
    .rename(columns={"series_id": "unique_id", "date": "ds", "value": "y"})
    .sort_values(["unique_id", "ds"])
    .reset_index(drop=True)
)

FEAST_REPO_PATH: D:\Portofolio Data science\Time Series\Explainable_AI_Forecast_and_explain_the_Unemployment_of_USA\2_data_processing\feature_store\feast_repo\feature_repo
feature_store.yaml exists: True
Using date as the event timestamp. To specify a column explicitly, please name it event_timestamp.


# Préparation des données

## Passage en format WIDE

In [21]:
ts_raw = (
    ts_raw
    .pivot(index="date", columns="series_id", values="value")
    .sort_index()
)

print(ts_raw)

series_id                  UNRATE
date                             
1960-01-01 00:00:00+00:00    -0.8
1960-02-01 00:00:00+00:00    -1.1
1960-03-01 00:00:00+00:00    -0.2
1960-04-01 00:00:00+00:00     0.0
1960-05-01 00:00:00+00:00     0.0
...                           ...
2025-05-01 00:00:00+00:00     0.2
2025-06-01 00:00:00+00:00     0.0
2025-07-01 00:00:00+00:00     0.0
2025-08-01 00:00:00+00:00     0.1
2025-09-01 00:00:00+00:00     0.3

[789 rows x 1 columns]


## Construire la série y

In [22]:
# Vérifie que l’index est bien une date (sinon essaie de le convertir)
y = ts_raw.copy()

if not isinstance(y.index, (pd.DatetimeIndex, pd.PeriodIndex)):
    y.index = pd.to_datetime(y.index, errors="coerce")

# aménager la fréquence mensuelle (début de mois)
y.index = y.index.to_period("M").to_timestamp(how="start")
y = y.sort_index().asfreq("MS").astype(float)

# 🔒 borne la date max (sans dropna)
y = y.loc[:pd.Timestamp("2025-08-01")]

# y (DataFrame) → Series 1D
if isinstance(y, pd.DataFrame):
    y = y.iloc[:, 0]

print(
    f"✅ Série prête : {y.index.min().date()} → {y.index.max().date()} "
    f"| n={len(y)} | freq={y.index.freqstr}"
)

✅ Série prête : 1960-01-01 → 2025-08-01 | n=788 | freq=MS


C:\Users\Mita\AppData\Local\Temp\ipykernel_11528\4267632309.py:8: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  y.index = y.index.to_period("M").to_timestamp(how="start")


# Run_config

In [23]:
# ==========================================
# AR(1) — Pseudo-OOS continu (h=12), p=1 fixe + BAGGING (bootstrap en blocs)
# ==========================================
# ---------- Paramètres ----------
h = 12
min_train_n = 36
trend = "c"
p_fixed = 1

# Conformal style
PI_WINDOWS = 3
STEP_SIZE = 12
LEVEL = 95
alpha = 1 - LEVEL/100
calib_size = PI_WINDOWS * STEP_SIZE   # 36
min_calib  = calib_size

## Boostrap Utilities

In [24]:
# ---------- Utilitaires bootstrap ----------
def moving_block_bootstrap(arr, L, rng):
    """Concatène des blocs contigus de taille L tirés aléatoirement jusqu'à longueur n."""
    n = len(arr)
    if L <= 0 or L > n:
        raise ValueError("L_block invalide")
    nb = int(np.ceil(n / L))
    starts = rng.integers(0, n - L + 1, size=nb)
    out = np.concatenate([arr[s:s+L] for s in starts])[:n]
    return out

def bagged_h_forecast_AR1(y_tr, h, trend, B, L, rng):
    """
    Prévision à horizon h par bagging (residual moving-block bootstrap) pour AR(1).
    Retourne (yhat_mean, yhat_dist, base_pred)
    """
    base_model = AutoReg(y_tr, lags=1, old_names=False, trend=trend).fit()
    base_fc = base_model.predict(start=len(y_tr), end=len(y_tr) + h - 1)
    base_pred = float(base_fc.iloc[-1])

    resid = base_model.resid.values
    fitted = (y_tr.iloc[-len(resid):].values - resid)  # ŷ_t aligné aux résidus

    boot_preds = []
    for _ in range(B):
        res_b = moving_block_bootstrap(resid, L, rng)   # bootstrap des résidus
        y_b = fitted + res_b                             # série bootstrapée
        m_b = AutoReg(pd.Series(y_b, index=y_tr.index[-len(y_b):]),
                      lags=1, old_names=False, trend=trend).fit()
        fc_b = m_b.predict(start=len(y_tr), end=len(y_tr) + h - 1)
        boot_preds.append(float(fc_b.iloc[-1]))
    return float(np.mean(boot_preds)), np.array(boot_preds), base_pred

In [25]:
# ---------- Sécurisation de la série y ----------
y = pd.Series(y.astype(float).values, index=pd.to_datetime(y.index)).asfreq("MS").dropna()
print(f"y: {y.index.min().date()} → {y.index.max().date()}  (n={len(y)}) | freq={y.index.freqstr}")

y: 1960-01-01 → 2025-08-01  (n=788) | freq=MS


In [26]:
rows = []
last_model = None
last_fit_end = None

# ✅ erreurs signées PAR horizon (1..12)
past_err_by_h = {k: [] for k in range(1, h+1)}

last_t_end = y.index.max() - relativedelta(months=h)

for t_end in y.index:
    if t_end > last_t_end:
        break

    y_tr = y.loc[:t_end]
    if len(y_tr) < max(min_train_n, p_fixed + 1):
        continue

    # fit mensuel
    ar1 = AutoReg(y_tr, lags=p_fixed, old_names=False, trend=trend).fit()
    last_model = ar1
    last_fit_end = t_end

    # forecast 1..12
    fc = ar1.predict(start=len(y_tr), end=len(y_tr) + h - 1)  # len = h

    # pour chaque horizon k = 1..12
    for k in range(1, h+1):
        t_fore = t_end + relativedelta(months=k)
        if t_fore not in y.index:
            continue

        yhat_k = float(fc.iloc[k-1])
        y_true = float(y.loc[t_fore])

        # --- conformal interval (distribution) POUR horizon k ---
        errs_k = past_err_by_h[k]
        if len(errs_k) >= min_calib:
            window = np.asarray(errs_k[-calib_size:])
            err_lo = float(np.quantile(window, alpha/2))
            err_hi = float(np.quantile(window, 1 - alpha/2))
            y_lo = yhat_k + err_lo
            y_hi = yhat_k + err_hi
        else:
            y_lo = np.nan
            y_hi = np.nan

        # update erreurs horizon k
        errs_k.append(y_true - yhat_k)

        rows.append((
            t_end,     # cutoff
            t_fore,    # date cible
            k,         # horizon
            y_true,    # obs
            yhat_k,    # pred
            y_lo,      # lo95
            y_hi       # hi95
        ))

# ---------- DataFrame long ----------
df_oos_long = pd.DataFrame(
    rows,
    columns=["cutoff", "date", "h", "y_obs", "y_hat_ar", "y_hat_ar_lo_95", "y_hat_ar_hi_95"]
).sort_values(["date", "h", "cutoff"]).reset_index(drop=True)

print("\n✅ OOS long terminé")
print(df_oos_long.head())

# ---------- Si tu veux le même format que StatsForecast pour plot_series ----------
# Pour imiter ce que plot_series attend, on peut garder tous les horizons,
# mais souvent tu veux un seul horizon (ex: h=12).
# ➜ filtre h=12 pour avoir une série de prévisions à 12 mois.
df_ar_forecasts = (
    df_oos_long[df_oos_long["h"] == 12]
    .assign(series_id="UNRATE")
    [["series_id", "date", "cutoff", "y_obs", "y_hat_ar", "y_hat_ar_lo_95", "y_hat_ar_hi_95"]]
    .sort_values(["series_id", "date"])
    .reset_index(drop=True)
)

print("\n✅ df_ar_forecasts (h=12) prêt pour plot_series")
print(df_ar_forecasts.head())


✅ OOS long terminé
      cutoff       date  h  y_obs  y_hat_ar  y_hat_ar_lo_95  y_hat_ar_hi_95
0 1962-12-01 1963-01-01  1   -0.1 -0.446879             NaN             NaN
1 1963-01-01 1963-02-01  1    0.4 -0.068810             NaN             NaN
2 1962-12-01 1963-02-01  2    0.4 -0.398027             NaN             NaN
3 1963-02-01 1963-03-01  1    0.1  0.401057             NaN             NaN
4 1963-01-01 1963-03-01  2    0.1 -0.040257             NaN             NaN

✅ df_ar_forecasts (h=12) prêt pour plot_series
  series_id       date     cutoff  y_obs  y_hat_ar  y_hat_ar_lo_95  \
0    UNRATE 1963-12-01 1962-12-01    0.0 -0.080890             NaN   
1    UNRATE 1964-01-01 1963-01-01   -0.1  0.141077             NaN   
2    UNRATE 1964-02-01 1963-02-01   -0.5  0.408114             NaN   
3    UNRATE 1964-03-01 1963-03-01   -0.3  0.242637             NaN   
4    UNRATE 1964-04-01 1963-04-01   -0.4  0.238955             NaN   

   y_hat_ar_hi_95  
0             NaN  
1             N

In [27]:
len(rows[0]), rows[0]

(7,
 (Timestamp('1962-12-01 00:00:00'),
  Timestamp('1963-01-01 00:00:00'),
  1,
  -0.09999999999999964,
  -0.44687855150325473,
  nan,
  nan))

In [28]:
# ---------- DataFrame OOS (LONG: cutoff x horizon) ----------
if rows:
    df_oos_ar1 = (
        pd.DataFrame(
            rows,
            columns=[
                "cutoff",
                "date",
                "h",
                "y_true",
                "y_hat",
                "y_hat_lo_95",
                "y_hat_hi_95",
            ],
        )
        .sort_values(["cutoff", "h", "date"])
        .reset_index(drop=True)
    )
else:
    df_oos_ar1 = pd.DataFrame(
        columns=[
            "cutoff", "date", "h",
            "y_true", "y_hat",
            "y_hat_lo_95", "y_hat_hi_95",
        ]
    )

# Normaliser au mois (début de mois)
df_oos_ar1["cutoff"] = (
    pd.to_datetime(df_oos_ar1["cutoff"], errors="coerce")
      .dt.to_period("M")
      .dt.to_timestamp(how="start")
)
df_oos_ar1["date"] = (
    pd.to_datetime(df_oos_ar1["date"], errors="coerce")
      .dt.to_period("M")
      .dt.to_timestamp(how="start")
)

print(f"\n✅ Pseudo-OOS terminé — n lignes = {len(df_oos_ar1)}")
print(df_oos_ar1.head(5))
print("\nDernières lignes :")
print(df_oos_ar1.tail(5))


✅ Pseudo-OOS terminé — n lignes = 8892
      cutoff       date  h  y_true     y_hat  y_hat_lo_95  y_hat_hi_95
0 1962-12-01 1963-01-01  1    -0.1 -0.446879          NaN          NaN
1 1962-12-01 1963-02-01  2     0.4 -0.398027          NaN          NaN
2 1962-12-01 1963-03-01  3     0.1 -0.353101          NaN          NaN
3 1962-12-01 1963-04-01  4     0.1 -0.311787          NaN          NaN
4 1962-12-01 1963-05-01  5     0.4 -0.273793          NaN          NaN

Dernières lignes :
         cutoff       date   h  y_true     y_hat  y_hat_lo_95  y_hat_hi_95
8887 2024-08-01 2025-04-01   8     0.3  0.207640    -0.839379     0.892549
8888 2024-08-01 2025-05-01   9     0.2  0.185746    -0.827589     0.854198
8889 2024-08-01 2025-06-01  10     0.0  0.166078    -0.718921     0.816778
8890 2024-08-01 2025-07-01  11     0.0  0.148409    -0.444550     0.837898
8891 2024-08-01 2025-08-01  12     0.1  0.132535    -0.244764     0.808580


In [29]:
# ---------- (facultatif) Scores par période ----------
if len(df_oos_ar1):
    df_val  = df_oos_ar1.loc["1983-01-01":"1989-12-31"].copy()
    df_test = df_oos_ar1.loc["1990-01-01":"2025-08-31"].copy()

    if len(df_val):
        mae  = mean_absolute_error(df_val["y_true"], df_val["y_hat"])
        rmse = np.sqrt(mean_squared_error(df_val["y_true"], df_val["y_hat"]))
        r2   = r2_score(df_val["y_true"], df_val["y_hat"]) if len(df_val) > 1 else np.nan
        print(f"\n📊 Validation 83–89 — n={len(df_val)} | MAE={mae:.3f} | RMSE={rmse:.3f} | R²={r2:.3f}")

    if len(df_test):
        mae  = mean_absolute_error(df_test["y_true"], df_test["y_hat"])
        rmse = np.sqrt(mean_squared_error(df_test["y_true"], df_test["y_hat"]))
        r2   = r2_score(df_test["y_true"], df_test["y_hat"]) if len(df_test) > 1 else np.nan
        print(f"📊 Test 90–2025 — n={len(df_test)} | MAE={mae:.3f} | RMSE={rmse:.3f} | R²={r2:.3f}")


📊 Validation 83–89 — n=6 | MAE=0.272 | RMSE=0.325 | R²=-0.637
📊 Test 90–2025 — n=35 | MAE=0.315 | RMSE=0.388 | R²=-1.016


In [30]:
# ---------- Sauvegardes (version CI) ----------
AR1_LAST_PKL  = "AR1_CI_last_trained_model.pkl"
AR1_LAST_META = "AR1_CI_last_trained_model_meta.csv"
AR1_BUNDLE    = "AR1_CI_h12_oos_bundle.pkl"

In [31]:
# ============================================================
# Sauvegardes — AR(1) + Conformal CI (LONG: cutoff x horizon)
# ============================================================
import pickle
import pandas as pd

# ---------- Noms de fichiers (version CI) ----------
AR1_LAST_PKL  = "AR1_CI_last_trained_model.pkl"
AR1_LAST_META = "AR1_CI_last_trained_model_meta.csv"
AR1_BUNDLE    = "AR1_CI_h12_oos_bundle.pkl"

# ============================================================
# ✅ Sécurités notebook (évite NameError si cellules exécutées dans le désordre)
# ============================================================
last_model = globals().get("last_model", None)
last_fit_end = globals().get("last_fit_end", None)

if df_oos_ar1 is None or (hasattr(df_oos_ar1, "empty") and df_oos_ar1.empty):
    raise RuntimeError("df_oos_ar1 est vide/introuvable. Exécute d'abord la cellule pseudo-OOS.")

# ============================================================
# 1) Sauvegarde du modèle final AR(1)
# ============================================================
if last_model is not None:
    try:
        import joblib  # ✅ évite NameError
        joblib.dump(last_model, AR1_LAST_PKL)
        print(f"💾 Modèle AR(1) sauvegardé → {AR1_LAST_PKL}")
    except Exception:
        with open(AR1_LAST_PKL, "wb") as f:
            pickle.dump(last_model, f)
        print(f"💾 Modèle AR(1) sauvegardé (pickle) → {AR1_LAST_PKL}")
else:
    print("⚠️ last_model est None (modèle final non sauvegardé).")


# ============================================================
# 2) Bundle des sorties (inclut CI conformal)
#    IMPORTANT: df_oos_ar1 est LONG (cutoff, date, h, ...)
# ============================================================
bundle = {
    "oos_predictions": (
        df_oos_ar1
        .copy()
        .rename(columns={
            "y_hat": "y_pred",
            "y_hat_lo_95": "y_pred_lo_95",
            "y_hat_hi_95": "y_pred_hi_95",
        })
        .assign(
            cutoff=lambda d: (
                pd.to_datetime(d["cutoff"], errors="coerce")
                  .dt.to_period("M")
                  .dt.to_timestamp(how="start")
            ),
            date=lambda d: (
                pd.to_datetime(d["date"], errors="coerce")
                  .dt.to_period("M")
                  .dt.to_timestamp(how="start")
            )
        )
    ),
    "params": {
        "model": "AR(1)",
        "trend": trend,
        "horizon": h,
        "lag": 1,
        "min_train_n": min_train_n,

        # ✅ CI info
        "ci_method": "conformal_distribution",
        "ci_level": 95,
        "ci_step_size": 12,
        "ci_windows": 3,
    },
    "meta": {
        "trained_until": str(last_fit_end.date()) if last_fit_end is not None else None,
        "index_freq": "MS",
        "n_obs_y": int(len(y)),
        "n_forecasts": int(len(df_oos_ar1)),
        "n_forecasts_with_ci": int(
            df_oos_ar1["y_hat_lo_95"].notna().sum()
            if "y_hat_lo_95" in df_oos_ar1.columns else 0
        ),
    }
}

with open(AR1_BUNDLE, "wb") as f:
    pickle.dump(bundle, f)

print(f"💾 Bundle AR(1) OOS sauvegardé → {AR1_BUNDLE}")


# ============================================================
# 3) Méta CSV (inclut CI)
# ============================================================
meta_row = {
    "model": "AR(1)",
    "trend": trend,
    "lag": 1,
    "horizon": h,
    "min_train_n": min_train_n,
    "trained_until": str(last_fit_end.date()) if last_fit_end is not None else None,
    "n_obs_y": int(len(y)),
    "n_forecasts": int(len(df_oos_ar1)),
    "ci_method": "conformal_distribution",
    "ci_level": 95,
    "ci_step_size": 12,
    "ci_windows": 3,
    "n_forecasts_with_ci": int(
        df_oos_ar1["y_hat_lo_95"].notna().sum()
        if "y_hat_lo_95" in df_oos_ar1.columns else 0
    ),
}

pd.DataFrame([meta_row]).to_csv(AR1_LAST_META, index=False)
print(f"💾 Méta AR(1) sauvegardée → {AR1_LAST_META}")

💾 Modèle AR(1) sauvegardé → AR1_CI_last_trained_model.pkl
💾 Bundle AR(1) OOS sauvegardé → AR1_CI_h12_oos_bundle.pkl
💾 Méta AR(1) sauvegardée → AR1_CI_last_trained_model_meta.csv
